# 3. SQL Data Ingestion

This notebook loads the clustered EMR data into PostgreSQL for Vanna to query.

In [ ]:
import os
import sys
import pandas as pd
from sqlalchemy import create_engine

sys.path.append(os.path.abspath('..'))
from src.config import settings

In [ ]:
print("1. Loading Clustered Data...")
file_path = os.path.join("..", "output", "clustered_emr.csv")
if not os.path.exists(file_path):
    raise FileNotFoundError("clustered_emr.csv not found. Run 1_clustering.ipynb first.")

df = pd.read_csv(file_path)
print(f"Loaded {len(df)} records.")

In [ ]:
print("2. Cleaning and Formatting Data for SQL...")

# Clean column names to be SQL-friendly (lowercase, underscores)
df.columns = [
    c.strip().lower().replace(' / ', '_').replace(' ', '_').replace('/', '_') 
    for c in df.columns
]

# Ensure date columns are proper datetime objects
date_cols = ['created_date', 'closed_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Drop combined_text as we don't need it in the structured DB
if 'combined_text' in df.columns:
    df = df.drop(columns=['combined_text'])

print("Columns ready for SQL:", df.columns.tolist())

In [ ]:
print("3. Connecting to PostgreSQL and Ingesting Data...")
engine = create_engine(settings.postgres_url)

table_name = 'emr_records'
print(f"Writing to table '{table_name}'...")

# Write data to postgres
df.to_sql(table_name, engine, if_exists='replace', index=False)
print("Ingestion complete.")

In [ ]:
print("4. Testing SQL Query...")
test_query = f"SELECT machine_model, COUNT(*) as fault_count FROM {table_name} GROUP BY machine_model ORDER BY fault_count DESC LIMIT 5"
result = pd.read_sql(test_query, engine)
print(result)